# Grid Sampling

This sampling is inspired by stratified grid sampling in ray tracing. 

Here we divide the geographical map coordinates into N by N grid (N=50 in this case), and uniform randomly pick K=10 samples within each grid cell. If there are less than K samples in a grid, all of the samples are taken. 

In [13]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
import os

In [14]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [15]:

if IN_COLAB:
    drive.mount('/content/drive')
else:
    print("Not running in Colab, skipping Drive mount.")

Not running in Colab, skipping Drive mount.


In [16]:
# Some globals

PROJECT_PATH = '.'
if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/CENG574_PROJECT'
    
DATA_PATH = os.path.join(PROJECT_PATH, 'data')

INPUT_PARQUET = os.path.join(DATA_PATH, "cleaned_partitions")
OUTPUT_PARQUET = os.path.join(DATA_PATH, "nyc_taxi_grid_sampled.parquet")



In [17]:

# =====================================================
# PARAMETERS
# =====================================================

N_GRID = 50      # NxN spatial grid
MAX_PER_CELL = 10

# =====================================================
# NYC BOUNDING BOX
# =====================================================

LAT_MIN = 40.5774
LAT_MAX = 40.9176

LON_MIN = -74.15
LON_MAX = -73.7004



In [18]:
# =====================================================
# LOAD
# =====================================================

print("Loading merged parquet...")
USE_DASK= True
try:
  print("Reading using dask...")
  ddf = dd.read_parquet(INPUT_PARQUET)
except:
  USE_DASK= False
  print("Reading using pandas...")
  ddf = pd.read_parquet(INPUT_PARQUET)

print("Computing sampled dataframe...")

# Convert to pandas
if USE_DASK:
  df = ddf.compute()
else:
  df =ddf

print(f"Original samples: {len(df):,}")

Loading merged parquet...
Reading using dask...
Computing sampled dataframe...
Original samples: 12,367,735


In [19]:


# =====================================================
# FILTER TO NYC BOUNDS
# =====================================================

df = df[
    (df["pickup_latitude"] >= LAT_MIN) &
    (df["pickup_latitude"] <= LAT_MAX) &
    (df["pickup_longitude"] >= LON_MIN) &
    (df["pickup_longitude"] <= LON_MAX)
]

print(f"After coordinate filtering: {len(df):,}")

# =====================================================
# GRID ASSIGNMENT
# =====================================================

lat_step = (LAT_MAX - LAT_MIN) / N_GRID
lon_step = (LON_MAX - LON_MIN) / N_GRID

df["grid_y"] = (
    ((df["pickup_latitude"] - LAT_MIN) / lat_step)
    .astype(int)
    .clip(0, N_GRID - 1)
)

df["grid_x"] = (
    ((df["pickup_longitude"] - LON_MIN) / lon_step)
    .astype(int)
    .clip(0, N_GRID - 1)
)


After coordinate filtering: 12,367,735


In [20]:

# =====================================================
# SAMPLE EACH CELL
# =====================================================

sampled_parts = []

for _, group in df.groupby(["grid_x", "grid_y"]):

    if len(group) > MAX_PER_CELL:
        sampled_parts.append(
            group.sample(MAX_PER_CELL, random_state=42)
        )
    else:
        sampled_parts.append(group)

sampled_df = pd.concat(sampled_parts, ignore_index=True)

# Remove helper columns
sampled_df = sampled_df.drop(
    columns=["grid_x", "grid_y"]
)

print(f"Final sampled size: {len(sampled_df):,}")
print(f"Final data set features (might include ground truth): {sampled_df.shape[1]}")

Final sampled size: 8,716
Final data set features (might include ground truth): 16


In [21]:

# =====================================================
# SAVE
# =====================================================

sampled_df.to_parquet(
    OUTPUT_PARQUET,
    index=False
)

print(f"Saved to {OUTPUT_PARQUET}")

Saved to .\data\nyc_taxi_grid_sampled.parquet
